# Superstore Sales Analysis
**Objective:** Analyze purchase volume, monthly seasonality, category-wise revenue, regional/customer-segment performance, and discount sensitivity.

**Tools:** Python, Pandas, Matplotlib, Seaborn.

**Dataset note:** The supplied CSV contains 10,800 lines, but only 9,994 rows are complete transaction records. The remaining lines are appended auxiliary/text rows with missing order dates and sales. These non-transaction rows are excluded rather than imputed. The cleaning steps below are reproducible from the original CSV.

## Business questions
1. How do sales and profit vary over time?
2. Which categories, regions, and customer segments contribute the most sales and profit?
3. How is purchase volume distributed across categories?
4. What relationship exists between discounts and profit?
5. What actions should management consider based on the evidence?

In [ ]:
# If needed, install dependencies in your environment:
# %pip install pandas matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Put superstore_sales.csv in the same folder as this notebook.
DATA_PATH = Path("superstore_sales.csv")
df_raw = pd.read_csv(DATA_PATH)

print("Raw shape:", df_raw.shape)
display(df_raw.head())
print("\\nMissing values in raw file:")
display(df_raw.isna().sum().sort_values(ascending=False).to_frame("Missing values").head(10))

## 1. Data cleaning and preparation

- Parse `Order Date` and `Ship Date` as dates.
- Remove non-transaction rows that do not contain an order date or sales value. The provided CSV has appended auxiliary rows after the transaction table.
- Remove exact duplicate records.
- Create `Year-Month` for time-series aggregation.
- Keep valid zero/negative profits: these are meaningful business outcomes, not missing data.
- Check missing values after cleaning. Postal code can legitimately be missing for some valid records, so it is not filled with an invented value.

In [ ]:
df = df_raw.copy()

# Parse date columns; invalid dates become NaT.
for col in ["Order Date", "Ship Date"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Exclude appended non-transaction rows lacking essential transaction fields.
before_rows = len(df)
df = df.dropna(subset=["Order Date", "Sales"]).copy()
after_required_filter = len(df)

# Remove exact duplicate transaction rows, if any.
before_dedup = len(df)
df = df.drop_duplicates().copy()
after_dedup = len(df)

# Ensure numeric analysis fields are numeric.
for col in ["Sales", "Profit", "Discount", "Quantity"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop records that cannot be analyzed due to invalid essential numeric values.
df = df.dropna(subset=["Sales", "Profit", "Discount", "Quantity", "Category", "Region", "Segment"])
df["Year-Month"] = df["Order Date"].dt.to_period("M").astype(str)
df["Order Year"] = df["Order Date"].dt.year
df["Order Month"] = df["Order Date"].dt.month

print(f"Rows in supplied file: {before_rows:,}")
print(f"Rows after removing rows without Order Date/Sales: {after_required_filter:,}")
print(f"Exact duplicates removed: {before_dedup - after_dedup:,}")
print(f"Final clean rows: {len(df):,}")
print(f"Date range: {df['Order Date'].min():%Y-%m-%d} to {df['Order Date'].max():%Y-%m-%d}")
print("\\nMissing values after cleaning:")
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing values").head(10))
display(df.head())

## 2. KPI overview

In [ ]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
total_orders = df["Order ID"].nunique()
total_customers = df["Customer ID"].nunique()
total_units = df["Quantity"].sum()
profit_margin = total_profit / total_sales * 100 if total_sales else np.nan

kpis = pd.DataFrame({
    "Metric": ["Total Sales", "Total Profit", "Profit Margin", "Unique Orders", "Unique Customers", "Units Sold"],
    "Value": [total_sales, total_profit, f"{profit_margin:.2f}%", total_orders, total_customers, total_units]
})
display(kpis)

## 3. Category-wise revenue and profit

In [ ]:
category_summary = (
    df.groupby("Category", as_index=False)
      .agg(Sales=("Sales", "sum"),
           Profit=("Profit", "sum"),
           Units=("Quantity", "sum"),
           Avg_Discount=("Discount", "mean"),
           Transactions=("Order ID", "count"))
      .sort_values("Sales", ascending=False)
)
display(category_summary.style.format({
    "Sales": "${:,.2f}", "Profit": "${:,.2f}",
    "Units": "{:,.0f}", "Avg_Discount": "{:.1%}", "Transactions": "{:,.0f}"
}))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=category_summary, x="Category", y="Sales", ax=axes[0])
axes[0].set_title("Total Sales by Product Category")
axes[0].set_ylabel("Sales")
axes[0].tick_params(axis="x", rotation=15)
sns.barplot(data=category_summary, x="Category", y="Profit", ax=axes[1])
axes[1].set_title("Total Profit by Product Category")
axes[1].set_ylabel("Profit")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

## 4. Monthly sales velocity and seasonality

In [ ]:
monthly = (
    df.groupby("Year-Month", as_index=False)
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Units=("Quantity", "sum"))
      .sort_values("Year-Month")
)
display(monthly.head())

plt.figure(figsize=(15, 5))
sns.lineplot(data=monthly, x="Year-Month", y="Sales", marker="o", label="Sales")
plt.title("Monthly Sales Trend")
plt.xlabel("Year-Month")
plt.ylabel("Sales")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()

# Average sales by calendar month helps compare recurring seasonality across years.
seasonality = (
    df.assign(Calendar_Month=df["Order Date"].dt.month)
      .groupby("Calendar_Month", as_index=False)
      .agg(Avg_Monthly_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum"))
)
# Normalize by number of years represented for each calendar month.
years_per_month = df.groupby(df["Order Date"].dt.month)["Order Year"].nunique()
seasonality["Avg_Sales_Per_Year"] = seasonality.apply(
    lambda r: r["Avg_Monthly_Sales"] / years_per_month.loc[r["Calendar_Month"]], axis=1
)
display(seasonality.sort_values("Avg_Sales_Per_Year", ascending=False))

plt.figure(figsize=(10, 4))
sns.barplot(data=seasonality, x="Calendar_Month", y="Avg_Sales_Per_Year")
plt.title("Average Sales per Year by Calendar Month")
plt.xlabel("Calendar month (1=Jan, 12=Dec)")
plt.ylabel("Average sales per year")
plt.tight_layout()
plt.show()

## 5. Region and customer segment performance

In [ ]:
region_summary = (
    df.groupby("Region", as_index=False)
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Units=("Quantity", "sum"))
      .sort_values("Sales", ascending=False)
)
segment_summary = (
    df.groupby("Segment", as_index=False)
      .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Units=("Quantity", "sum"))
      .sort_values("Sales", ascending=False)
)
print("Region summary")
display(region_summary)
print("Customer segment summary")
display(segment_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=region_summary, x="Region", y="Sales", ax=axes[0])
axes[0].set_title("Sales by Region")
sns.barplot(data=segment_summary, x="Segment", y="Sales", ax=axes[1])
axes[1].set_title("Sales by Customer Segment")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

## 6. Purchase volume by category and segment

In [ ]:
volume_pivot = pd.pivot_table(
    df, index="Category", columns="Segment", values="Quantity",
    aggfunc="sum", fill_value=0
)
display(volume_pivot)

plt.figure(figsize=(9, 5))
sns.heatmap(volume_pivot, annot=True, fmt=".0f", cmap="Blues")
plt.title("Units Sold: Category × Customer Segment")
plt.xlabel("Customer segment")
plt.ylabel("Product category")
plt.tight_layout()
plt.show()

## 7. Discount sensitivity: discount versus profit

Discount bands are used alongside the scatter plot to make the pattern easier to interpret. The relationship is descriptive and does **not** prove discounts alone caused a profit change; category mix, product cost, and order size may also matter.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="Discount", y="Profit", hue="Category", alpha=0.55)
plt.axhline(0, color="black", linewidth=1)
plt.title("Discount vs. Profit by Product Category")
plt.xlabel("Discount rate")
plt.ylabel("Profit")
plt.tight_layout()
plt.show()

# Use fixed discount intervals, including no discount.
bins = [-0.001, 0.001, 0.10, 0.20, 0.30, 0.40, 0.501]
labels = ["~0%", ">0–10%", ">10–20%", ">20–30%", ">30–40%", ">40–50%"]
df["Discount Band"] = pd.cut(df["Discount"], bins=bins, labels=labels, include_lowest=True)
discount_summary = (
    df.groupby("Discount Band", observed=False)
      .agg(Transactions=("Profit", "size"),
           Sales=("Sales", "sum"),
           Profit=("Profit", "sum"),
           Mean_Profit=("Profit", "mean"),
           Mean_Discount=("Discount", "mean"))
      .reset_index()
)
display(discount_summary.style.format({
    "Sales": "${:,.2f}", "Profit": "${:,.2f}",
    "Mean_Profit": "${:,.2f}", "Mean_Discount": "{:.1%}"
}))

plt.figure(figsize=(10, 5))
sns.barplot(data=discount_summary, x="Discount Band", y="Profit")
plt.axhline(0, color="black", linewidth=1)
plt.title("Total Profit by Discount Band")
plt.xlabel("Discount band")
plt.ylabel("Total profit")
plt.tight_layout()
plt.show()

## 8. Aggregate analysis by category, region, and segment

In [ ]:
grouped_analysis = (
    df.groupby(["Category", "Region", "Segment"], as_index=False)
      .agg(Sales=("Sales", "sum"),
           Profit=("Profit", "sum"),
           Units=("Quantity", "sum"),
           Average_Discount=("Discount", "mean"),
           Orders=("Order ID", "nunique"))
      .sort_values("Sales", ascending=False)
)
display(grouped_analysis.head(15))
# Full grouped output can be exported for further exploration.
grouped_analysis.to_csv("superstore_category_region_segment_summary.csv", index=False)
print("Saved: superstore_category_region_segment_summary.csv")

## 9. Executive takeaways (5+ evidence-based findings)

The following findings are generated from the cleaned data, so they update if the dataset changes. Treat them as descriptive business insights rather than proof of causation.

In [ ]:
# Executive findings generated from the analysis
top_sales_category = category_summary.sort_values("Sales", ascending=False).iloc[0]
top_profit_category = category_summary.sort_values("Profit", ascending=False).iloc[0]
lowest_profit_category = category_summary.sort_values("Profit", ascending=True).iloc[0]
top_region = region_summary.sort_values("Sales", ascending=False).iloc[0]
top_segment = segment_summary.sort_values("Sales", ascending=False).iloc[0]
peak_month = monthly.sort_values("Sales", ascending=False).iloc[0]
discount_band_profit = discount_summary.dropna(subset=["Profit"])
worst_discount_band = discount_band_profit.sort_values("Profit").iloc[0]
best_discount_band = discount_band_profit.sort_values("Profit", ascending=False).iloc[0]

findings = [
    f"1. **Revenue leader:** {top_sales_category['Category']} is the largest sales category, generating ${top_sales_category['Sales']:,.2f} in sales. Protect availability and review its product-level contribution.",
    f"2. **Profit leader:** {top_profit_category['Category']} contributes the most total profit (${top_profit_category['Profit']:,.2f}). Prioritize profitable assortment and cross-selling in this category.",
    f"3. **Category margin warning:** {lowest_profit_category['Category']} has the lowest total profit (${lowest_profit_category['Profit']:,.2f}) despite ${lowest_profit_category['Sales']:,.2f} in sales. Review its costs, returns, product mix, and discount policy.",
    f"4. **Regional opportunity:** {top_region['Region']} leads regional sales with ${top_region['Sales']:,.2f}. Compare its winning products and customer mix with lower-sales regions before reallocating inventory or marketing budgets.",
    f"5. **Customer segment:** {top_segment['Segment']} is the largest segment by sales (${top_segment['Sales']:,.2f}). Test retention and basket-building campaigns for this segment, while evaluating profitability as well as revenue.",
    f"6. **Seasonality:** The highest-sales month in the dataset is {peak_month['Year-Month']} (${peak_month['Sales']:,.2f}). Plan stock and staffing ahead of recurring peaks, checking the month-of-year chart to distinguish seasonality from one-off spikes.",
    f"7. **Discount sensitivity:** The {worst_discount_band['Discount Band']} discount band has the lowest total profit (${worst_discount_band['Profit']:,.2f}); the {best_discount_band['Discount Band']} band has the highest (${best_discount_band['Profit']:,.2f}). Review high-discount deals and use controlled tests rather than blanket discounts."
]
for finding in findings:
    print(finding)

## 10. Recommendations and limitations

### Recommended actions
- Evaluate product-level profit and margin, not revenue alone, when setting category priorities.
- Forecast inventory and staffing using the monthly trend and recurring calendar-month pattern.
- Investigate high-discount orders and consider discount approval thresholds where profit becomes negative.
- Compare region and segment performance using both sales and profit before deciding where to invest.
- Run controlled discount experiments and account for product/category mix before claiming a causal effect.

### Limitations
- This is historical observational data; it does not establish why sales or profit changed.
- Discount bands aggregate different products and order sizes, so their results may be affected by mix.
- Missing postal codes are retained as missing rather than guessed.
- The supplied file includes appended auxiliary rows; the notebook excludes rows without valid order date and sales.
- All currency values are reported in the dataset's native currency units.